In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/AIMO3_Reference_Problems.pdf
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/sample_submission.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_inference_server.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/__init__.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/templates.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/base_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/relay.py
/kaggle/input/comp

In [2]:
import os
import re
import sympy as sp

In [3]:
COMPETITION_DIR_CANDIDATES = [
    "/kaggle/input/ai-mathematical-olympiad-progress-prize-3",
    "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3",
]

ANSWER_MOD = 100000
DEBUG = True

In [4]:
def resolve_competition_dir():
    for path in COMPETITION_DIR_CANDIDATES:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        "Competition input directory was not found. "
        "Please check the actual path under /kaggle/input."
    )

competition_dir = resolve_competition_dir()

if DEBUG:
    print("Competition directory:", competition_dir)
    print("Files:", os.listdir(competition_dir))

test_path = os.path.join(competition_dir, "test.csv")
sample_submission_path = os.path.join(competition_dir, "sample_submission.csv")

Competition directory: /kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3
Files: ['reference.csv', 'AIMO3_Reference_Problems.pdf', 'sample_submission.csv', 'test.csv', 'kaggle_evaluation']


In [5]:
test_df = pd.read_csv(test_path)

if DEBUG:
    print("test.csv shape:", test_df.shape)
    print(test_df.head())

test.csv shape: (3, 2)
       id                 problem
0  000aaa          What is $1-1$?
1  111bbb    What is $0\times10$?
2  222ccc  Solve $4+x=4$ for $x$.


In [6]:
def extract_boxed_number(text: str):
    if not isinstance(text, str):
        return None
    
    match = re.search(r'\\boxed\{(-?\d+)\}', text)
    if match:
        return int(match.group(1)) % ANSWER_MOD
    
    return None

def solve_with_sympy_if_easy(problem: str):
    if not isinstance(problem, str):
        return 0
    
    # Very conservative baseline:
    # only solve extremely simple one-variable polynomial equations if detected
    try:
        cleaned = problem.strip()

        # Example pattern:
        # "Solve x^2 - 5x + 6 = 0"
        simple_eq_match = re.search(r'([xX][\s\S]*?)=\s*0', cleaned)
        if simple_eq_match and len(cleaned) < 200:
            x = sp.symbols('x')
            expr_text = simple_eq_match.group(1)

            expr_text = expr_text.replace("^", "**")
            expr_text = re.sub(r'(\d)([xX])', r'\1*\2', expr_text)
            expr_text = expr_text.replace("X", "x")

            expr = sp.sympify(expr_text)
            roots = sp.solve(sp.Eq(expr, 0), x)

            integer_roots = []
            for r in roots:
                if r.is_integer:
                    integer_roots.append(int(r))

            if integer_roots:
                return min(integer_roots) % ANSWER_MOD

    except Exception:
        pass

    # Safe fallback answer
    return 0

def predict_answer(problem: str):
    # This is a placeholder baseline.
    # Replace this function with a stronger solver later.
    return solve_with_sympy_if_easy(problem)


In [7]:
submission_df = pd.DataFrame()
submission_df["id"] = test_df["id"]

problem_column = None
for candidate in ["problem", "question", "prompt"]:
    if candidate in test_df.columns:
        problem_column = candidate
        break

if problem_column is None:
    raise ValueError(
        "No problem text column was found. "
        "Expected one of: problem, question, prompt"
    )

submission_df["answer"] = test_df[problem_column].apply(predict_answer).astype(int) % ANSWER_MOD
submission_df["answer"] = submission_df["answer"].clip(0, ANSWER_MOD - 1)

In [8]:
output_path = "/kaggle/working/submission.csv"
submission_df.to_csv(output_path, index=False)

print("Saved submission file to:", output_path)
print(submission_df.head())

Saved submission file to: /kaggle/working/submission.csv
       id  answer
0  000aaa       0
1  111bbb       0
2  222ccc       0
